# Point-in-time features
Predict at 2011-09-01 using strictly earlier records. Future labels require a complete 90-day observation window.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT))
from src.features.build import make_features,make_targets,make_sequences
from src.utils.config import config
cfg=config()
ledger=pd.read_parquet(ROOT/'data/interim/transactions.parquet')
features=make_features(ledger,cfg['cutoff'],cfg)
features.head()

In [ ]:
targets=make_targets(ledger,features.customer_id,cfg['cutoff'],cfg['horizon_days'])
targets.describe(include='all')

In [ ]:
before=ledger[ledger.invoice_date<pd.Timestamp(cfg['cutoff'])]
pd.testing.assert_frame_equal(features,make_features(before,cfg['cutoff'],cfg))
sequence,lengths=make_sequences(ledger,features.customer_id.head(5),cfg['cutoff'],12)
sequence.shape,lengths

## Interpret carefully
Historical CLV is signed net spend. Predicted CLV is future gross spend over 90 days. Product groups and discount sensitivity are proxies. Read `docs/feature_dictionary.md` for exact formulas.